# 🎓 AI Classroom Engagement Tracker
## YOLOv8 Object Detection — Complete Training Notebook
**Classes:** `attentive` · `distracted` · `drowsy`

This notebook trains a **YOLOv8s** object detection model that detects students in classroom
images and video and classifies each person's engagement state using transfer learning from
COCO-pretrained weights.

> ✏️ **Only the ⚙️ CONFIG cell needs editing.** Set your dataset path, then run **Runtime → Run all**.

---
| # | Section |
|---|---|
| 1 | [Installation & Setup](#1) |
| 2 | [⚙️ Configuration](#2) |
| 3 | [Dataset Validation & data.yaml](#3) |
| 4 | [Training](#4) |
| 5 | [Evaluation — Precision / Recall / mAP](#5) |
| 6 | [Inference — Images & Video](#6) |
| 7 | [Improvement Tips](#7) |

---
## 1 · Installation & Setup <a id="1"></a>
Install **Ultralytics** (which bundles YOLOv8) and verify that a GPU is available.
If running locally, make sure you have already run `pip install -r requirements.txt`.

In [1]:
import sys, os
from pathlib import Path

# Detect Google Colab
IN_COLAB = 'google.colab' in sys.modules
DRIVE_MOUNTED = False

if IN_COLAB:
    # Mount Google Drive
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=True)
        DRIVE_MOUNTED = True
        print('✅ Google Drive mounted at /content/drive')
    except Exception as e:
        print(f'⚠️  Drive mount failed: {e}')
        print()
        print('  Your options:')
        print('  A) Restart: Runtime → Restart session, re-run this cell,')
        print('     and click Allow in the Google sign-in popup.')
        print()
        print('  B) Skip Drive and upload your dataset directly to Colab:')
        print('     - Click the 📁 folder icon in the left sidebar')
        print('     - Drag and drop your dataset zip into /content/')
        print('     - Then unzip it:')
        print('       !unzip /content/dataset.zip -d /content/dataset')
        print()
        print('  ⚡ Using /content/ as working directory for now.')

    print('\n📦 Installing Ultralytics (YOLOv8)...')
    ret = os.system('pip install ultralytics -q')
    if ret == 0:
        print('✅ Ultralytics installed.\n')
    else:
        print('❌ Install failed — check your internet connection.\n')

# Suggest default paths based on mount status
if DRIVE_MOUNTED:
    SUGGESTED_DATASET = '/content/drive/MyDrive/Students-YoloV8'
    SUGGESTED_YAML    = '/content/drive/MyDrive/data.yaml'
    print('📌 Suggested CONFIG paths (copy into the CONFIG cell below):')
    print(f'   dataset_path : \'{SUGGESTED_DATASET}\'')
    print(f'   data_yaml    : \'{SUGGESTED_YAML}\'')
else:
    SUGGESTED_DATASET = '/content/Students-YoloV8'
    SUGGESTED_YAML    = '/content/data.yaml'
    print('📌 Suggested CONFIG paths for local Colab storage:')
    print(f'   dataset_path : \'{SUGGESTED_DATASET}\'')
    print(f'   data_yaml    : \'{SUGGESTED_YAML}\'')

# Verify GPU
import torch
print(f'\nPyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {props.total_memory / 1e9:.1f} GB')
else:
    print()
    print('⚠️  No GPU found — training will be very slow on CPU.')
    print('   Fix: Runtime → Change runtime type → Hardware accelerator → T4 GPU')

Mounted at /content/drive
✅ Google Drive mounted at /content/drive

📦 Installing Ultralytics (YOLOv8)...
✅ Ultralytics installed.

📌 Suggested CONFIG paths (copy into the CONFIG cell below):
   dataset_path : '/content/drive/MyDrive/Students-YoloV8'
   data_yaml    : '/content/drive/MyDrive/data.yaml'

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


---
## 2 · ⚙️ Configuration <a id="2"></a>

**This is the only cell you need to edit.** Every path and hyperparameter used by the
rest of the notebook is read from `CONFIG` below.

| Parameter | Recommended value | When to change |
|---|---|---|
| `dataset_path` | `/content/dataset` | Always — set your actual path |
| `model_weights` | `yolov8s.pt` | Use `yolov8m` if accuracy is still weak after 100 epochs |
| `epochs` | `50` | Increase to `100` if mAP is still rising at epoch 50 |
| `imgsz` | `640` | Increase to `1280` if students appear tiny in wide-angle shots |
| `batch` | `16` | Reduce to `8` if you see CUDA out-of-memory errors |
| `lr0` | `0.01` | Lower to `0.005` only if training loss oscillates wildly |
| `patience` | `20` | Increase to `30` for small datasets to avoid stopping too early |

In [2]:
import yaml, csv, time, shutil
import cv2
import torch
from pathlib import Path
from ultralytics import YOLO

# ⚙️  CONFIGURATION
CONFIG = {

    'dataset_path' : '/content/drive/MyDrive/Students-YoloV8',
    'data_yaml'    : '/content/drive/MyDrive/data.yaml',

    # MODEL
    'model_weights': 'yolov8m.pt',

    # TRAINING HYPERPARAMETERS
    'epochs'       : 100,     # Full passes through the training set.
    'imgsz'        : 640,     # Input resolution (px, square).  YOLOv8 standard.
    'batch'        : 16,      # Images per gradient update.  Reduce to 8 for OOM.
    'lr0'          : 0.01,    # Initial learning rate (cosine decay schedule).
    'lrf'          : 0.01,    # LR multiplier at final epoch (lr0 * lrf = final LR).
    'momentum'     : 0.937,   # SGD momentum — YOLOv8 default, leave it.
    'weight_decay' : 0.0005,  # L2 regularisation to discourage overfitting.
    'patience'     : 30,      # Early-stop if val mAP does not improve for N epochs.

    # ACCURACY & CONFIDENCE FIXES
    'label_smoothing': 0.1,

    'copy_paste'     : 0.3,

    'cls'            : 0.5,

    # DATA AUGMENTATION
    'mosaic'    : 1.0,   # Stitch 4 images together — boosts diversity a lot.
    'mixup'     : 0.15,  # Blend 2 images softly — mild regulariser.
    'fliplr'    : 0.5,   # Horizontal flip 50% of the time (safe for classrooms).
    'flipud'    : 0.0,   # Vertical flip — keep at 0 (classrooms are upright).
    'hsv_h'     : 0.015, # Hue variation — helps with different lighting conditions.
    'hsv_s'     : 0.7,   # Saturation variation.
    'hsv_v'     : 0.4,   # Brightness variation.
    'degrees'   : 10.0,  # Random rotation ± 10°.
    'translate' : 0.1,   # Random translation ± 10% of image size.
    'scale'     : 0.5,   # Random scale ± 50%.

    # HARDWARE & OUTPUT
    'device'      : 'cuda' if torch.cuda.is_available() else 'cpu',
    'workers'     : 4,      # Dataloader threads.  Use 0 on Windows if errors.
    'cache'       : False,  # Cache images in RAM (faster but needs more memory).
    'save_period' : 10,     # Save extra checkpoint every N epochs.
    'project'     : '/content/runs/train',
    'name'        : 'classroom_engagement_v2',
}

# Class names (order must match label indices in .txt files)
CLASS_NAMES  = ['attentive', 'distracted', 'drowsy']

# Colour palette for bounding boxes — BGR (OpenCV format)
# Green = attentive, Orange = distracted, Red = drowsy
CLASS_COLORS = {
    'attentive'  : (50,  200,  50),
    'distracted' : (0,   165, 255),
    'drowsy'     : (50,   50, 220),
}

print('✅ Configuration loaded.')
print(f'   Device   : {CONFIG["device"].upper()}')
print(f'   Model    : {CONFIG["model_weights"]}')
print(f'   Epochs   : {CONFIG["epochs"]}')
print(f'   Img size : {CONFIG["imgsz"]} px')
print(f'   Batch    : {CONFIG["batch"]}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Configuration loaded.
   Device   : CUDA
   Model    : yolov8m.pt
   Epochs   : 100
   Img size : 640 px
   Batch    : 16


---
## 3 · Dataset Validation & data.yaml <a id="3"></a>

### Expected folder structure
```
dataset/
├── images/
│   ├── train/   ← .jpg / .png files used for training
│   ├── val/     ← .jpg / .png files used for validation during training
│   └── test/    ← .jpg / .png files used only for final evaluation
└── labels/
    ├── train/   ← one .txt file per image (YOLO format)
    ├── val/
    └── test/
```

### YOLO label format (per `.txt` file)
Each line = one bounding box:
```
<class_id>  <x_centre>  <y_centre>  <width>  <height>
```
All values are normalised to `[0.0, 1.0]` relative to image size.

**Example line:** `0 0.512 0.345 0.180 0.420`
→ class=`attentive`, centred at (51.2 %, 34.5 %), box width=18 %, height=42 %.

In [3]:
# Validate dataset folder structure
def validate_dataset(dataset_path: str) -> bool:
    '''
    Auto-detects whether the dataset uses standard structure (images/train/)
    or Roboflow structure (train/images/), checks all folders, and prints
    a clear status table.  Returns True only if everything looks correct.
    '''
    base = Path(dataset_path)
    print('📂 Checking dataset folder structure...')
    print(f'   Root: {base.resolve()}\n')

    # Auto-detect which layout is being used
    roboflow_layout = (base / 'train' / 'images').exists()
    standard_layout = (base / 'images' / 'train').exists()

    if roboflow_layout:
        print('   📦 Detected: Roboflow export layout (train/images/, valid/images/)')
        required = [
            'train/images', 'train/labels',
            'valid/images', 'valid/labels',
            'test/images',  'test/labels',
        ]
    elif standard_layout:
        print('   📦 Detected: Standard layout (images/train/, labels/train/)')
        required = [
            'images/train', 'images/val', 'images/test',
            'labels/train', 'labels/val', 'labels/test',
        ]
    else:
        print('   ❌ Could not detect dataset layout.')
        print('   Expected either  train/images/  OR  images/train/  inside the root.')
        return False

    img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    txt_exts = {'.txt'}
    all_ok   = True

    print()
    for rel in required:
        full   = base / rel
        is_lbl = 'labels' in rel
        exts   = txt_exts if is_lbl else img_exts
        count  = len([f for f in full.glob('*') if f.suffix.lower() in exts]) if full.exists() else 0
        ok     = full.exists() and count > 0
        icon   = '✅' if ok else '❌'
        if not ok:
            all_ok = False
        unit = 'labels' if is_lbl else 'images'
        print(f'   {icon}  {rel:<30}  ({count} {unit})')

    print()
    if all_ok:
        print('✅ Dataset structure looks correct.\n')
    else:
        print('❌ One or more required folders are missing or empty.')
        print('   Fix the folders above before running training.\n')
    return all_ok


# Create / validate data.yaml
def ensure_data_yaml(config: dict) -> str:

    yaml_path   = Path(config['data_yaml'])
    dataset_abs = str(Path(config['dataset_path']).resolve())

    if yaml_path.exists():
        print(f'✅ Found data.yaml at: {yaml_path}')
        with open(yaml_path) as f:
            data = yaml.safe_load(f)

        # 'val' and 'valid' are both accepted by YOLOv8 — only flag truly missing keys
        required_keys = ['train', 'nc', 'names']
        missing = [k for k in required_keys if k not in data]
        if missing:
            raise ValueError(
                f'data.yaml is missing required keys: {missing}\n'
                f'Please check the file at {yaml_path}'
            )

        # Warn if path key is missing — YOLOv8 needs it for relative sub-paths
        if 'path' not in data:
            print('   ⚠️  data.yaml has no "path:" key.')
            print('      If train/val/test values are relative, add:')
            print(f'      path: {dataset_abs}')

        nc    = data['nc']
        names = data['names']
        print(f'   nc    : {nc}')
        print(f'   names : {names}\n')
        return str(yaml_path)

    # Auto-generate if missing
    base   = Path(dataset_abs)
    is_rbf = (base / 'train' / 'images').exists()   # Roboflow layout?

    print(f'⚠️  data.yaml not found — auto-generating at {yaml_path}...')
    content = {
        'path' : dataset_abs,
        'train': 'train/images'  if is_rbf else 'images/train',
        'val'  : 'valid/images'  if is_rbf else 'images/val',
        'test' : 'test/images',
        'nc'   : 3,
        'names': {0: 'distracted', 1: 'drowsy', 2: 'attentive'},
    }
    yaml_path.parent.mkdir(parents=True, exist_ok=True)
    with open(yaml_path, 'w') as f:
        yaml.dump(content, f, default_flow_style=False, sort_keys=False)
    print(f'✅ data.yaml written ({"Roboflow" if is_rbf else "standard"} layout).\n')
    return str(yaml_path)


# Run checks
dataset_ok = validate_dataset(CONFIG['dataset_path'])
DATA_YAML  = ensure_data_yaml(CONFIG)
print(f'DATA_YAML set to: {DATA_YAML}')

📂 Checking dataset folder structure...
   Root: /content/drive/MyDrive/Students-YoloV8

   📦 Detected: Roboflow export layout (train/images/, valid/images/)

   ✅  train/images                    (1131 images)
   ✅  train/labels                    (1131 labels)
   ✅  valid/images                    (108 images)
   ✅  valid/labels                    (108 labels)
   ✅  test/images                     (53 images)
   ✅  test/labels                     (53 labels)

✅ Dataset structure looks correct.

✅ Found data.yaml at: /content/drive/MyDrive/data.yaml
   nc    : 3
   names : {0: 'attentive', 1: 'distracted', 2: 'drowsy'}

DATA_YAML set to: /content/drive/MyDrive/data.yaml


---
## 4 · Training <a id="4"></a>

### Why these choices?

**Transfer learning:** YOLOv8s is pretrained on 80 MS-COCO classes, so its backbone already
detects people, shapes, and textures. Only the detection head is re-trained for our 3 classes.
This gives good accuracy with as few as ~300 images per class.

**Mosaic augmentation** stitches 4 random training images into one composite image, dramatically
increasing scene diversity and teaching the model to handle partial occlusions — exactly the
kind of overlap you see in crowded classroom rows.

**Early stopping** (`patience=20`) halts training automatically if validation mAP has not
improved for 20 consecutive epochs, saving GPU time and preventing overfitting.

**Cosine LR decay** starts at `lr0=0.01` and smoothly decays to `lr0 × lrf = 0.0001`,
which lets the model make large updates early and fine-tune carefully at the end.

In [4]:
# Define training function
def train_model(config: dict, data_yaml: str):
    '''
    Loads a COCO-pretrained YOLOv8 model and fine-tunes it on the
    classroom engagement dataset.
    '''
    print('=' * 60)
    print('  AI Classroom Engagement Tracker — Training Start')
    print('=' * 60)
    print(f'  Device    : {config["device"].upper()}')
    print(f'  Model     : {config["model_weights"]}')
    print(f'  Epochs    : {config["epochs"]}')
    print(f'  Batch     : {config["batch"]}')
    print(f'  Img size  : {config["imgsz"]} px')
    print(f'  Classes   : attentive | distracted | drowsy')
    print('=' * 60 + '\n')

    # Load pretrained weights (downloaded automatically on first run)
    model = YOLO(config['model_weights'])
    print(f'✅ Loaded base model: {config["model_weights"]}\n')

    results = model.train(
        data         = data_yaml,
        epochs       = config['epochs'],
        imgsz        = config['imgsz'],
        batch        = config['batch'],
        lr0          = config['lr0'],
        lrf          = config['lrf'],
        momentum     = config['momentum'],
        weight_decay = config['weight_decay'],
        patience     = config['patience'],
        # Augmentation
        mosaic       = config['mosaic'],
        mixup        = config['mixup'],
        fliplr       = config['fliplr'],
        flipud       = config['flipud'],
        hsv_h        = config['hsv_h'],
        hsv_s        = config['hsv_s'],
        hsv_v        = config['hsv_v'],
        degrees      = config['degrees'],
        translate    = config['translate'],
        scale        = config['scale'],
        # Accuracy & confidence fixes — v2 additions
        label_smoothing = config['label_smoothing'],  # Softens training targets → less overconfidence
        copy_paste      = config['copy_paste'],        # Paste objects across images → pose variety
        cls             = config['cls'],               # Classification loss weight
        # Hardware
        device       = config['device'],
        workers      = config['workers'],
        cache        = config['cache'],
        # Output
        project      = config['project'],
        name         = config['name'],
        save_period  = config['save_period'],
        pretrained   = True,   # Always use COCO weights as starting point
        verbose      = True,
        plots        = True,   # Save loss curves, PR curve, confusion matrix
        exist_ok     = True,
    )
    return model, results

In [5]:
# Run training
if not dataset_ok:
    print('❌ Fix the dataset structure above before training.')
else:
    trained_model, train_results = train_model(CONFIG, DATA_YAML)

    ACTUAL_RUN_DIR = Path(trained_model.trainer.save_dir)
    CONFIG['_actual_run_dir'] = str(ACTUAL_RUN_DIR)

    print('\n✅ Training finished.')
    print(f'   Weights saved to: {ACTUAL_RUN_DIR / "weights" / "best.pt"}')

  AI Classroom Engagement Tracker — Training Start
  Device    : CUDA
  Model     : yolov8m.pt
  Epochs    : 100
  Batch     : 16
  Img size  : 640 px
  Classes   : attentive | distracted | drowsy

✅ Loaded base model: yolov8m.pt

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.64 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, im

In [6]:
# Post-training summary

run_dir = ACTUAL_RUN_DIR   # set by the training cell above

print('\n' + '=' * 60)
print('  📁  SAVED FILES')
print('=' * 60)
print(f'  Best weights  : {run_dir}/weights/best.pt')
print(f'  Last weights  : {run_dir}/weights/last.pt')
print(f'  Training log  : {run_dir}/results.csv')
print(f'  Loss curves   : {run_dir}/results.png')
print(f'  Confusion mat : {run_dir}/confusion_matrix.png')
print(f'  PR curve      : {run_dir}/PR_curve.png')
print(f'  F1 curve      : {run_dir}/F1_curve.png')
print('=' * 60)

# Update the weights path variables used by evaluation and inference cells
BEST_PT = str(Path(run_dir) / 'weights' / 'best.pt')
print(f'\n  BEST_PT = {BEST_PT}')
print('  (This variable is used automatically by the evaluation and inference cells)')

# Parse results.csv for final epoch metrics
results_csv = Path(run_dir) / 'results.csv'
if results_csv.exists():
    with open(results_csv) as f:
        rows = list(csv.DictReader(f))
    if rows:
        last = {k.strip(): v.strip() for k, v in rows[-1].items()}
        print('\n  📋  FINAL EPOCH METRICS (validation set):')
        print(f'     Precision  : {float(last.get("metrics/precision(B)", 0)):.4f}')
        print(f'     Recall     : {float(last.get("metrics/recall(B)", 0)):.4f}')
        print(f'     mAP @ 0.50 : {float(last.get("metrics/mAP50(B)", 0)):.4f}')
        print(f'     mAP@50-95  : {float(last.get("metrics/mAP50-95(B)", 0)):.4f}')
else:
    print('\n  ⚠️  results.csv not found — training may not have completed.')


  📁  SAVED FILES
  Best weights  : /content/runs/train/classroom_engagement_v2/weights/best.pt
  Last weights  : /content/runs/train/classroom_engagement_v2/weights/last.pt
  Training log  : /content/runs/train/classroom_engagement_v2/results.csv
  Loss curves   : /content/runs/train/classroom_engagement_v2/results.png
  Confusion mat : /content/runs/train/classroom_engagement_v2/confusion_matrix.png
  PR curve      : /content/runs/train/classroom_engagement_v2/PR_curve.png
  F1 curve      : /content/runs/train/classroom_engagement_v2/F1_curve.png

  BEST_PT = /content/runs/train/classroom_engagement_v2/weights/best.pt
  (This variable is used automatically by the evaluation and inference cells)

  📋  FINAL EPOCH METRICS (validation set):
     Precision  : 0.9850
     Recall     : 0.9767
     mAP @ 0.50 : 0.9863
     mAP@50-95  : 0.8306


In [7]:
# Quick validation on val split using best.pt
# Reloads the best checkpoint and runs a quick sanity-check on the val set.

if not Path(BEST_PT).exists():
    print(f'⚠️  best.pt not found at: {BEST_PT}')
    print('   Run the training cell above first.')
else:
    print(f'🔍 Loading best.pt: {BEST_PT}')
    best_model = YOLO(BEST_PT)

    val_results = best_model.val(
        data    = DATA_YAML,
        split   = 'val',
        imgsz   = CONFIG['imgsz'],
        batch   = CONFIG['batch'],
        device  = CONFIG['device'],
        verbose = True,
    )
    print('\n  mAP@50    :', round(val_results.box.map50, 4))
    print('  mAP@50-95 :', round(val_results.box.map,   4))
    print('  Precision :', round(val_results.box.mp,    4))
    print('  Recall    :', round(val_results.box.mr,    4))

🔍 Loading best.pt: /content/runs/train/classroom_engagement_v2/weights/best.pt
Ultralytics 8.4.64 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 16.9±7.4 MB/s, size: 46.4 KB)
val: Scanning /content/drive/MyDrive/Students-YoloV8/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 108/108 32.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6it/s 4.3s
                   all        108        168      0.985      0.987      0.987      0.839
             attentive         44         54          1      0.981      0.985      0.892
            distracted         49         53      0.987          1      0.995      0.809
                drowsy         52         61      0.968      0.979       0.98      0.817
Speed: 5.3ms preprocess, 23.9ms inference

---
## 5 · Evaluation <a id="5"></a>

Evaluation runs on the **held-out test set** — images the model has never seen during
training or validation.  YOLOv8 computes the standard COCO detection metrics:

| Metric | What it means |
|---|---|
| **Precision** | Of all predicted boxes, what fraction are correct? |
| **Recall** | Of all ground-truth boxes, what fraction were found? |
| **mAP@50** | Mean Average Precision at IoU threshold 0.50 (primary metric) |
| **mAP@50-95** | mAP averaged over IoU 0.50 → 0.95 (stricter COCO benchmark) |

A large gap between mAP@50 and mAP@50-95 means boxes are detected but imprecisely
localised — see Section 7 for fixes.

In [8]:
# Load best weights for evaluation

if 'BEST_PT' not in dir() or not Path(BEST_PT).exists():
    print('⚠️  BEST_PT is not set or the file does not exist.')
    print('   Either run the training section first, or set BEST_PT manually:')
    print('   BEST_PT = \'<path to your best.pt>\'')
else:
    eval_model = YOLO(BEST_PT)
    print(f'✅ Loaded: {BEST_PT}')

✅ Loaded: /content/runs/train/classroom_engagement_v2/weights/best.pt


In [9]:
# Run evaluation on the TEST set

eval_results = eval_model.val(
    data      = DATA_YAML,
    split     = 'test',          # Use the held-out test split
    imgsz     = CONFIG['imgsz'],
    batch     = CONFIG['batch'],
    conf      = 0.001,           # Low threshold for complete PR curve
    iou       = 0.6,
    device    = CONFIG['device'],
    plots     = True,            # Saves confusion_matrix.png, PR_curve.png, F1_curve.png
    project   = './runs/evaluate',
    name      = 'test_set',
    exist_ok  = True,
    verbose   = True,
)

print('\n✅ Evaluation complete.')
print(f'   Plots saved to: ./runs/evaluate/test_set/')

Ultralytics 8.4.64 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 0.1±0.0 MB/s, size: 47.1 KB)
val: Scanning /content/drive/MyDrive/Students-YoloV8/test/labels.cache... 53 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 53/53 18.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 11.2s/it 44.8s
                   all         53         81      0.965      0.964      0.991      0.853
             attentive         20         30      0.987      0.967      0.994      0.916
            distracted         21         24      0.976          1      0.995      0.831
                drowsy         20         27      0.932      0.926      0.985      0.813
Speed: 5.1ms preprocess, 25.7ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to /content/runs/detect/ru

In [10]:
# ── Print full metrics table ──────────────────────────────────────────────────
box = eval_results.box  # DetMetrics object — contains all detection metrics

# ap_idx maps row positions in box.p / box.r / box.ap50 / box.ap to class IDs
ap_idx = box.ap_class_index.tolist() if hasattr(box.ap_class_index, 'tolist') else list(box.ap_class_index)

# ── Overall metrics ───────────────────────────────────────────────────────────
print('\n' + '=' * 62)
print('  📊  OVERALL METRICS  (Test Set)')
print('=' * 62)
print(f'  {"Precision (mean)":<26} {box.mp:>10.4f}')
print(f'  {"Recall (mean)":<26} {box.mr:>10.4f}')
print(f'  {"mAP @ IoU 0.50":<26} {box.map50:>10.4f}')
print(f'  {"mAP @ IoU 0.50-0.95":<26} {box.map:>10.4f}')
print('=' * 62)

# ── Per-class breakdown: Precision / Recall / mAP50 / mAP50-95 ───────────────
print('\n  📋  PER-CLASS BREAKDOWN')
print('=' * 62)
print(f'  {"Class":<14}  {"Precision":>10}  {"Recall":>8}  {"mAP@50":>8}  {"mAP50-95":>10}')
print(f'  {"-"*14}  {"-"*10}  {"-"*8}  {"-"*8}  {"-"*10}')
for i, cls_i in enumerate(ap_idx):
    name = CLASS_NAMES[cls_i] if cls_i < len(CLASS_NAMES) else f'class_{cls_i}'
    p    = float(box.p[i])    if i < len(box.p)    else 0.0
    r    = float(box.r[i])    if i < len(box.r)    else 0.0
    a50  = float(box.ap50[i]) if i < len(box.ap50) else 0.0
    a    = float(box.ap[i])   if i < len(box.ap)   else 0.0
    print(f'  {name:<14}  {p:>10.4f}  {r:>8.4f}  {a50:>8.4f}  {a:>10.4f}')
print('=' * 62)

# ── IoU & mAP50 detailed breakdown ───────────────────────────────────────────
# Shows exactly which IoU thresholds are used, so you can report them correctly
# in your write-up.  The bar chart makes the weakest class obvious at a glance.
print('\n  📐  IoU & mAP50 DETAILED BREAKDOWN')
print('=' * 62)
print(f'  IoU threshold for NMS (non-max suppression) : 0.60')
print(f'  IoU range for mAP computation               : 0.50 → 0.95  (step 0.05, 10 thresholds)')
print()
print(f'  mAP @ IoU 0.50            (primary metric)  : {box.map50:.4f}')

# mAP@75 — available in Ultralytics >= 8.2
try:
    map75 = float(box.map75)
    print(f'  mAP @ IoU 0.75            (stricter check)  : {map75:.4f}')
except AttributeError:
    pass   # Older Ultralytics versions do not expose map75

print(f'  mAP @ IoU 0.50-0.95       (COCO standard)   : {box.map:.4f}')
print()

# Per-class AP@IoU0.50 with a filled/empty bar so weak classes stand out
print(f'  {"Class":<14}  {"AP @ IoU 0.50":>14}  {"AP @ IoU 0.50-0.95":>20}  Visual (AP@50)')
print(f'  {"-"*14}  {"-"*14}  {"-"*20}  {"-"*22}')
for i, cls_i in enumerate(ap_idx):
    name = CLASS_NAMES[cls_i] if cls_i < len(CLASS_NAMES) else f'class_{cls_i}'
    a50  = float(box.ap50[i]) if i < len(box.ap50) else 0.0
    a    = float(box.ap[i])   if i < len(box.ap)   else 0.0
    filled = int(a50 * 20)
    bar    = '█' * filled + '░' * (20 - filled)
    print(f'  {name:<14}  {a50:>14.4f}  {a:>20.4f}  {bar}')
print('=' * 62)

# ── Plain-English interpretation ─────────────────────────────────────────────
map50 = float(box.map50)
print('\n  🔎  INTERPRETATION')
if map50 >= 0.80:
    print(f'  ✅ Excellent — mAP@50={map50:.3f} (≥ 0.80). Model is production-quality.')
elif map50 >= 0.65:
    print(f'  ✅ Good — mAP@50={map50:.3f} (≥ 0.65). Solid result for a university project.')
elif map50 >= 0.50:
    print(f'  ⚠️  Fair — mAP@50={map50:.3f}. Model works but see Section 7 for improvements.')
else:
    print(f'  ❌ Weak — mAP@50={map50:.3f} (< 0.50). Review Section 7 troubleshooting guide.')

gap = float(box.map50) - float(box.map)
print(f'\n  mAP@50 vs mAP@50-95 gap: {gap:.4f}')
if gap > 0.20:
    print('  ⚠️  Large gap — bounding boxes are imprecise. Check label quality or raise imgsz.')
else:
    print('  ✅ Small gap — box localisation is accurate.')


  📊  OVERALL METRICS  (Test Set)
  Precision (mean)               0.9650
  Recall (mean)                  0.9642
  mAP @ IoU 0.50                 0.9913
  mAP @ IoU 0.50-0.95            0.8534

  📋  PER-CLASS BREAKDOWN
  Class            Precision    Recall    mAP@50    mAP50-95
  --------------  ----------  --------  --------  ----------
  attentive           0.9874    0.9667    0.9940      0.9155
  distracted          0.9759    1.0000    0.9950      0.8312
  drowsy              0.9317    0.9259    0.9849      0.8135

  📐  IoU & mAP50 DETAILED BREAKDOWN
  IoU threshold for NMS (non-max suppression) : 0.60
  IoU range for mAP computation               : 0.50 → 0.95  (step 0.05, 10 thresholds)

  mAP @ IoU 0.50            (primary metric)  : 0.9913
  mAP @ IoU 0.75            (stricter check)  : 0.9560
  mAP @ IoU 0.50-0.95       (COCO standard)   : 0.8534

  Class            AP @ IoU 0.50    AP @ IoU 0.50-0.95  Visual (AP@50)
  --------------  --------------  --------------------  ---

---
## 6 · Inference <a id="6"></a>

Run the trained model on new classroom footage.
- **Images / folders:** detections are saved as annotated copies.
- **Video:** every frame is annotated and written to an `.mp4` output file.
  An engagement summary (attentive %, distracted %, drowsy %) is printed at the end.
- **Webcam:** live preview on your local machine (not available inside Colab).

Adjust `INFER_CONF` (0.25–0.45) to control the confidence threshold.
Higher = fewer but more reliable detections.

In [11]:
# Shared inference settings

INFER_CONF   = 0.45    # Confidence threshold — raised to 0.45 in v2 to filter weak predictions
INFER_IOU    = 0.45    # NMS IoU threshold
INFER_IMGSZ  = CONFIG['imgsz']
INFER_DEVICE = CONFIG['device']
INFER_SAVE   = '/content/runs/predict'   # Root output folder

if 'BEST_PT' not in dir() or not Path(BEST_PT).exists():
    print('⚠️  BEST_PT is not set or the file does not exist.')
    print('   Run the training section first, or set BEST_PT manually.')
else:
    infer_model = YOLO(BEST_PT)
    print(f'✅ Inference model loaded: {BEST_PT}')
    print(f'   Confidence threshold : {INFER_CONF}')
    print(f'   Device               : {INFER_DEVICE.upper()}')

✅ Inference model loaded: /content/runs/train/classroom_engagement_v2/weights/best.pt
   Confidence threshold : 0.45
   Device               : CUDA


In [12]:
# Helper: draw colour-coded bounding boxes
def draw_boxes(frame, result, line_width: int = 2):
    '''
    Draws CLASS_COLORS-coded boxes with class name + confidence onto a BGR frame.
    Returns the annotated frame (NumPy array).
    '''
    annotated = frame.copy()
    for box in result.boxes:
        cls_i    = int(box.cls[0])
        conf_val = float(box.conf[0])
        cls_name = CLASS_NAMES[cls_i] if cls_i < len(CLASS_NAMES) else f'cls_{cls_i}'
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        color = CLASS_COLORS.get(cls_name, (180, 180, 180))
        label = f'{cls_name}  {conf_val:.2f}'

        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, line_width)

        (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(annotated, (x1, y1 - th - bl - 4), (x1 + tw + 4, y1), color, -1)
        cv2.putText(annotated, label, (x1 + 2, y1 - bl - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    return annotated


def draw_hud(frame, counts: dict):
    '''
    Draws a semi-transparent engagement count overlay in the top-left corner.
    '''
    total = sum(counts.values()) or 1
    overlay = frame.copy()
    cv2.rectangle(overlay, (5, 5), (235, 105), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.45, frame, 0.55, 0, frame)
    lines = [
        (f'Attentive : {counts["attentive"]:>2} ({counts["attentive"]/total*100:4.1f}%)',  CLASS_COLORS['attentive']),
        (f'Distracted: {counts["distracted"]:>2} ({counts["distracted"]/total*100:4.1f}%)', CLASS_COLORS['distracted']),
        (f'Drowsy    : {counts["drowsy"]:>2} ({counts["drowsy"]/total*100:4.1f}%)',          CLASS_COLORS['drowsy']),
    ]
    for i, (text, color) in enumerate(lines):
        cv2.putText(frame, text, (10, 28 + i * 26),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 1)
    return frame

print('✅ Helper functions ready.')

✅ Helper functions ready.


In [13]:
# Inference on a single image or a folder of images

SOURCE_IMAGES = '/content/drive/MyDrive/Students-YoloV8/test/images'

if not Path(SOURCE_IMAGES).exists():
    print(f'❌ Path not found: {SOURCE_IMAGES}')
else:
    save_dir = Path(INFER_SAVE) / 'images'
    save_dir.mkdir(parents=True, exist_ok=True)

    results_list = infer_model.predict(
        source     = SOURCE_IMAGES,
        conf       = INFER_CONF,
        iou        = INFER_IOU,
        imgsz      = INFER_IMGSZ,
        device     = INFER_DEVICE,
        save       = True,        # Saves annotated copies automatically
        project    = INFER_SAVE,
        name       = 'images',
        exist_ok   = True,
        verbose    = False,
        line_width = 2,
    )

    # Print per-image summary
    total_det = {cls: 0 for cls in CLASS_NAMES}
    print(f'\n  Image{"":25}  Attentive  Distracted  Drowsy')
    print('  ' + '-' * 60)
    for r in results_list:
        img_counts = {cls: 0 for cls in CLASS_NAMES}
        for b in r.boxes:
            n = CLASS_NAMES[int(b.cls[0])] if int(b.cls[0]) < len(CLASS_NAMES) else 'unknown'
            if n in img_counts:
                img_counts[n] += 1
                total_det[n]  += 1
        name = Path(r.path).name[:30]
        print(f'  {name:<30} {img_counts["attentive"]:>9}  {img_counts["distracted"]:>10}  {img_counts["drowsy"]:>6}')

    print('  ' + '-' * 60)
    print(f'  {"TOTAL":<30} {total_det["attentive"]:>9}  {total_det["distracted"]:>10}  {total_det["drowsy"]:>6}')
    print(f'\n  ✅ Annotated images saved to: {(save_dir).resolve()}')

Results saved to /content/runs/predict/images

  Image                           Attentive  Distracted  Drowsy
  ------------------------------------------------------------
  14_jpg.rf.241b82738aa146c8c90e         0           0       1
  15_jpg.rf.34054200cbb176bcb17f         0           0       1
  192_jpg.rf.c479d57f2dba1f801c4         0           0       3
  195_jpg.rf.6389b2c4bffc0f6aae7         0           0       3
  210_jpg.rf.7a3f93b3dbd327c9862         0           1       0
  222_jpg.rf.826ca7f0536610b0c07         0           1       2
  348_jpg.rf.fbe985037d4b3514c96         2           0       0
  361_jpg.rf.f3e482314e8013f7442         0           2       0
  368_jpg.rf.12290ca2e9f7922143a         0           2       0
  3921_jpg.rf.9dfb1960650a22a257         1           1       0
  409_jpg.rf.4a105f6bf4592224bf1         0           1       1
  412_jpg.rf.e53510deeb323f83dc9         0           1       0
  415_jpg.rf.4a7e7ed1aa5fbd70c69         0           1       0
  418_j

In [14]:
# Inference on a video file

SOURCE_VIDEO = '/content/drive/MyDrive/video.mp4'

if not Path(SOURCE_VIDEO).exists():
    print(f'❌ Video not found: {SOURCE_VIDEO}')
    print('   Update SOURCE_VIDEO to a valid .mp4 / .avi / .mov path.')
else:
    in_path  = Path(SOURCE_VIDEO)
    out_dir  = Path(INFER_SAVE) / 'video'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{in_path.stem}_annotated.mp4'

    cap = cv2.VideoCapture(str(in_path))
    fps    = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    writer = cv2.VideoWriter(str(out_path),
                             cv2.VideoWriter_fourcc(*'mp4v'),
                             fps, (width, height))

    print(f'🎬 Processing: {in_path.name}')
    print(f'   {width}×{height}  |  {fps} FPS  |  {total} frames\n')

    agg      = {cls: 0 for cls in CLASS_NAMES}   # aggregate across all frames
    n_frames = 0
    t0       = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        n_frames += 1

        preds  = infer_model.predict(frame, conf=INFER_CONF, iou=INFER_IOU,
                                     imgsz=INFER_IMGSZ, device=INFER_DEVICE,
                                     verbose=False)
        r      = preds[0]
        counts = {cls: 0 for cls in CLASS_NAMES}
        for b in r.boxes:
            n = CLASS_NAMES[int(b.cls[0])] if int(b.cls[0]) < len(CLASS_NAMES) else None
            if n:
                counts[n] += 1
                agg[n]    += 1

        frame = draw_boxes(frame, r)
        frame = draw_hud(frame, counts)
        writer.write(frame)

        if n_frames % 50 == 0:
            elapsed = time.time() - t0
            pct     = n_frames / total * 100 if total else 0
            print(f'   Frame {n_frames:>5}/{total}  ({pct:5.1f}%)  —  {n_frames/elapsed:.1f} FPS')

    cap.release()
    writer.release()
    print(f'\n✅ Saved: {out_path.resolve()}  ({time.time()-t0:.1f}s total)\n')

    # Engagement summary
    grand = sum(agg.values()) or 1
    print('  📊  ENGAGEMENT SUMMARY')
    print('  ' + '─' * 44)
    for cls in CLASS_NAMES:
        pct = agg[cls] / grand * 100
        bar = '█' * int(pct / 4)
        print(f'  {cls:<12} {agg[cls]:>6} detections  ({pct:5.1f}%)  {bar}')

    score = (agg['attentive'] - agg['distracted'] - 0.5 * agg['drowsy']) / grand * 100
    print(f'\n  Engagement score: {score:+.1f} / 100')
    print('  (attentive=+1pt, drowsy=−0.5pt, distracted=−1pt per detection)')

❌ Video not found: /content/drive/MyDrive/video.mp4
   Update SOURCE_VIDEO to a valid .mp4 / .avi / .mov path.


In [15]:
# live webcam inference (local machine only, not Colab)

WEBCAM_INDEX = 0   # 0 for the built-in camera

if IN_COLAB:
    print('⚠️  Webcam inference is not supported inside Google Colab.')
    print('   Use the image or video cells above, or run this notebook locally.')
else:
    cap = cv2.VideoCapture(WEBCAM_INDEX)
    if not cap.isOpened():
        print(f'❌ Could not open webcam {WEBCAM_INDEX}')
    else:
        print('🎥 Webcam open — press  q  in the preview window to quit.')
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            preds  = infer_model.predict(frame, conf=INFER_CONF, iou=INFER_IOU,
                                         imgsz=INFER_IMGSZ, device=INFER_DEVICE,
                                         verbose=False)
            r      = preds[0]
            counts = {cls: 0 for cls in CLASS_NAMES}
            for b in r.boxes:
                n = CLASS_NAMES[int(b.cls[0])] if int(b.cls[0]) < len(CLASS_NAMES) else None
                if n:
                    counts[n] += 1
            frame = draw_boxes(frame, r)
            frame = draw_hud(frame, counts)
            cv2.imshow('Classroom Engagement Tracker  [q to quit]', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        cap.release()
        cv2.destroyAllWindows()
        print('✅ Webcam closed.')

⚠️  Webcam inference is not supported inside Google Colab.
   Use the image or video cells above, or run this notebook locally.
